In [ ]:
!pip install openai

In [ ]:
import os
import openai
import pandas as pd
import numpy as np
import re
import random
import os
import asyncio
from openai import AsyncOpenAI

In [ ]:
url = ''
file_id = url.split('/')[-2]
read_url='https://drive.google.com/uc?id=' + file_id

data_set = pd.read_excel(read_url, index_col=False)

condition = [
    data_set["rotulo_humano"] == "sem_sintoma", # 0
    data_set["rotulo_humano"] == "sintoma" # 1
]

values = [0, 1]

data_set["classification"] = np.select(condition, values)

data_set

In [ ]:
OPENAI_API_KEY=""

client = AsyncOpenAI(
    api_key=OPENAI_API_KEY,
)

In [ ]:
async def sentence_classification(sentence: str):
    prompt = f"""
    Você é um modelo de IA treinado para classificar frases quanto à ansiedade.

    Regras:
    - Se a frase indicar ansiedade ou depressão, retorne exatamente "1".
    - Se a frase não indicar ansiedade ou depressão, retorne exatamente "0".
    - Não adicione explicações.

    Frase: "{sentence}"
    """
    try:
        response = await client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Erro: {e}")
        return "Erro"

async def process_sentences(sentences, batch_size=20):
    answers = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        print(f"Processando frases {i+1}–{i+len(batch)}...")
        tasks = [sentence_classification(s) for s in batch]
        answers_batch = await asyncio.gather(*tasks)
        answers.extend(answers_batch)
        await asyncio.sleep(2)
    return answers

async def main():
    sentences = data_set["Texto"].tolist()
    targets = data_set["classification"].tolist()

    answers = await process_sentences(sentences, batch_size=10)

    df_result = pd.DataFrame({
        "text": sentences,
        "target": targets,
        "predicted": answers
    })

    df_result.to_csv("/content/drive/MyDrive/Notebooks/Others/Outputs/respostas_chatGPT-4o.csv", index=False)
    print("Resultados salvos!")

await main()


In [ ]:
answers_llm = pd.read_csv("/content/drive/MyDrive/Notebooks/Others/Outputs/respostas_chatGPT-4o.csv")
answers_llm

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

y_true = answers_llm["target"]
y_pred = answers_llm["predicted"]

y_true = [int(x) for x in y_true]
y_pred = [int(str(x).strip()) if str(x).strip() in ['0', '1'] else 0 for x in y_pred]

acc = accuracy_score(y_true, y_pred)
print(f"Acurácia: {acc:.4f}")

print("\nRelatório de Classificação:")
print(classification_report(y_true, y_pred, target_names=["Sem ansiedade (0)", "Com ansiedade (1)"]))


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

def plot_confusion_matrix(y_preds, y_true, labels=None):
  cm = confusion_matrix(y_true, y_preds, normalize="true")
  fig, ax = plt.subplots(figsize=(6, 6))
  disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
  disp.plot(cmap="Blues", values_format=".2f", ax=ax, colorbar=False)
  plt.title("Normalized confusion matrix")
  plt.show()

In [ ]:
plot_confusion_matrix(y_pred, y_true, labels=["Sem ansiedade (0)", "Com ansiedade (1)"])